# 09 · Evaluación de los modelos de lenguaje

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Inferencia de un modelo ajustado (mismo prompt del entrenamiento, decodificación voraz, máximo 6 tokens) y medición de calidad y eficiencia: latencia con lote 1 y lote 16, memoria GPU pico y tamaño del adaptador. De este notebook se toman las métricas de eficiencia; las métricas de calidad oficiales son las del notebook 10. Esta ejecución corresponde a Mistral 7B.

*Nombre durante el desarrollo: `07_Evaluacion_Final_Split_Oficial` (algunas salidas conservan esa numeración).*

## 1. Instalar dependencias

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes huggingface_hub scikit-learn pandas pyarrow rouge-score nltk
!pip uninstall -y -q torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency

## 2. Configuración

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"

MODELOS_DISPONIBLES = {
    "qwen_1_5b":  "Qwen/Qwen2.5-1.5B-Instruct",
    "llama_3b":   "meta-llama/Llama-3.2-3B-Instruct",
    "qwen_3b":    "Qwen/Qwen2.5-3B-Instruct",
    "phi35_mini": "microsoft/Phi-3.5-mini-instruct",
    "phi3_mini":  "microsoft/Phi-3-mini-4k-instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen_7b":    "Qwen/Qwen2.5-7B-Instruct",
    "llama_8b":   "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "gemma2_2b":  "google/gemma-2-2b-it",
    "gemma2_9b":  "google/gemma-2-9b-it",
}

CLAVE_MODELO = "mistral_7b"
MODEL_ID = MODELOS_DISPONIBLES[CLAVE_MODELO]

REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"
REPO_CHECKPOINTS = f"{TU_USUARIO_HF}/emotion-futbol-checkpoints"

COLUMNA_TEXTO = "text"
MAX_LENGTH = 112
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
N_MUESTRA_LATENCIA = 300

print(f"Evaluando: {CLAVE_MODELO} -> {MODEL_ID}")

Evaluando: mistral_7b -> mistralai/Mistral-7B-Instruct-v0.3


## 3. Descargar el conjunto de evaluación

In [3]:
from huggingface_hub import hf_hub_download
import pandas as pd

ruta_test = hf_hub_download(
    repo_id=REPO_DATOS,
    repo_type="dataset",
    filename="04b_validacion_final_combinada.csv",
)
df_test = pd.read_csv(ruta_test)
print(f"Test set final: {len(df_test)} comentarios")
print(df_test['label'].value_counts())

04b_validacion_final_combinada.csv:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Test set final: 10000 comentarios
label
alegria     3253
enojo       3000
burla       1678
neutral     1128
tristeza     478
sorpresa     427
miedo         36
Name: count, dtype: int64


## 4. Encontrar automáticamente el adaptador de este modelo

In [4]:
from huggingface_hub import list_repo_files
import re

archivos = list_repo_files(REPO_CHECKPOINTS, repo_type="dataset")

if any(f.startswith(f"{CLAVE_MODELO}/adapter_final/") for f in archivos):
    ruta_relativa_adaptador = f"{CLAVE_MODELO}/adapter_final"
    print(f"Usando el adaptador final de '{CLAVE_MODELO}' (entrenamiento completo, 3 épocas sobre 72k).")
else:
    pasos = sorted({
        int(m.group(1))
        for f in archivos
        if (m := re.match(rf"^{re.escape(CLAVE_MODELO)}/checkpoint-(\d+)/", f))
    })
    if not pasos:
        raise RuntimeError(
            f"No se encontró ni el adaptador final ni ningún checkpoint de '{CLAVE_MODELO}' en "
            f"{REPO_CHECKPOINTS}. ¿Ya empezó a entrenar este modelo?"
        )
    ultimo_paso = pasos[-1]
    ruta_relativa_adaptador = f"{CLAVE_MODELO}/checkpoint-{ultimo_paso}"
    print(f"AVISO: todavía no hay adaptador final para '{CLAVE_MODELO}' -- usando el checkpoint "
          f"intermedio más reciente (checkpoint-{ultimo_paso}). Estos resultados son PRELIMINARES, "
          f"no del modelo ya terminado.")

print(f"Ruta en la bodega de checkpoints: {ruta_relativa_adaptador}")

Usando el adaptador final de 'mistral_7b' (entrenamiento completo, 3 épocas sobre 72k).
Ruta en la bodega de checkpoints: mistral_7b/adapter_final


## 5. Cargar el modelo base + el adaptador LoRA entrenado

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import snapshot_download

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
)

ruta_adaptador = snapshot_download(
    repo_id=REPO_CHECKPOINTS,
    repo_type="dataset",
    allow_patterns=[f"{ruta_relativa_adaptador}/*"],
)
ruta_adaptador_local = f"{ruta_adaptador}/{ruta_relativa_adaptador}"

modelo = PeftModel.from_pretrained(modelo_base, ruta_adaptador_local)
modelo.eval()

print(f"Modelo + adaptador LoRA cargados: {CLAVE_MODELO} ({ruta_relativa_adaptador})")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Modelo + adaptador LoRA cargados: mistral_7b (mistral_7b/adapter_final)


## 6. Tamaño en disco del adaptador

In [6]:
import os

tamano_adaptador_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fnames in os.walk(ruta_adaptador_local)
    for f in fnames
) / (1024 ** 2)

print(f"Tamaño del adaptador LoRA de '{CLAVE_MODELO}' en disco: {tamano_adaptador_mb:.1f} MB")

Tamaño del adaptador LoRA de 'mistral_7b' en disco: 164.3 MB


## 7. Formato del prompt

In [7]:
def formatear_prompt_inferencia(texto):
    return (
        f"Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: "
        f"{', '.join(EMOCIONES)}.\n\nComentario: {texto}\n\nEmoción:"
    )

def parsear_emocion(texto_generado):
    limpio = texto_generado.strip().lower().split()[0] if texto_generado.strip() else ""
    limpio = limpio.strip(".,;:!¡¿?\"'")
    return limpio if limpio in EMOCIONES else "invalido"

print(formatear_prompt_inferencia("Qué golazo, no lo puedo creer"))

Clasifica la emoción del siguiente comentario de fútbol en una de estas categorías: alegria, tristeza, enojo, miedo, sorpresa, burla, neutral.

Comentario: Qué golazo, no lo puedo creer

Emoción:


## 8. Inferencia sobre los 10.000 comentarios del test set

In [8]:
from tqdm.auto import tqdm
import time

TAMANO_LOTE = 16
predicciones = []
predicciones_crudas = []

df_test = df_test.reset_index(drop=True)
prompts = [formatear_prompt_inferencia(t) for t in df_test[COLUMNA_TEXTO]]

inicio_lote = time.time()
for inicio in tqdm(range(0, len(prompts), TAMANO_LOTE), desc=f"Inferencia {CLAVE_MODELO}"):
    lote = prompts[inicio:inicio + TAMANO_LOTE]
    entradas = tokenizer(lote, return_tensors="pt", padding=True, truncation=True,
                          max_length=MAX_LENGTH).to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(
            **entradas,
            max_new_tokens=6,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    for i in range(len(lote)):
        generado = tokenizer.decode(
            salida[i][entradas['input_ids'].shape[1]:], skip_special_tokens=True
        )
        predicciones_crudas.append(generado)
        predicciones.append(parsear_emocion(generado))
torch.cuda.synchronize()
latencia_lote_prom_seg = (time.time() - inicio_lote) / len(prompts)

df_test['prediccion'] = predicciones
df_test['prediccion_cruda'] = predicciones_crudas
n_invalidas = (df_test['prediccion'] == 'invalido').sum()
print(f"Predicciones fuera de las 7 clases esperadas: {n_invalidas} de {len(df_test)} "
      f"({100 * n_invalidas / len(df_test):.2f}%)")
print(f"Latencia en lote (comparable con 08/09, lote={TAMANO_LOTE}): {latencia_lote_prom_seg:.4f} s/comentario")

Inferencia mistral_7b:   0%|          | 0/625 [00:00<?, ?it/s]

Predicciones fuera de las 7 clases esperadas: 1057 de 10000 (10.57%)
Latencia en lote (comparable con 08/09, lote=16): 1.0668 s/comentario


## 9. Métricas de calidad

In [9]:
from sklearn.metrics import accuracy_score, classification_report

reporte = classification_report(
    df_test['label'], df_test['prediccion'], labels=EMOCIONES,
    output_dict=True, zero_division=0
)
accuracy = accuracy_score(df_test['label'], df_test['prediccion'])

print(classification_report(df_test['label'], df_test['prediccion'], labels=EMOCIONES, zero_division=0))
print(f"Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

     alegria       0.87      0.81      0.84      3253
    tristeza       0.85      0.72      0.78       478
       enojo       0.81      0.74      0.77      3000
       miedo       0.58      0.50      0.54        36
    sorpresa       0.74      0.62      0.67       427
       burla       0.85      0.75      0.80      1678
     neutral       0.72      0.59      0.65      1128

   micro avg       0.83      0.74      0.78     10000
   macro avg       0.78      0.67      0.72     10000
weighted avg       0.82      0.74      0.78     10000

Accuracy: 0.7389


## 9b. ROUGE-L y BLEU

In [10]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
suavizado_bleu = SmoothingFunction().method1

rouge_l_por_ejemplo = []
bleu1_por_ejemplo = []

for etiqueta_real, texto_crudo in zip(df_test['label'], df_test['prediccion_cruda']):
    candidato = texto_crudo.strip().lower()
    referencia = etiqueta_real.strip().lower()

    rl = scorer_rouge.score(referencia, candidato)['rougeL'].fmeasure
    rouge_l_por_ejemplo.append(rl)

    if candidato:
        bleu1 = sentence_bleu([referencia.split()], candidato.split(), weights=(1.0,),
                               smoothing_function=suavizado_bleu)
    else:
        bleu1 = 0.0
    bleu1_por_ejemplo.append(bleu1)

df_test['rouge_l'] = rouge_l_por_ejemplo
df_test['bleu1'] = bleu1_por_ejemplo

rouge_l_promedio = sum(rouge_l_por_ejemplo) / len(rouge_l_por_ejemplo)
bleu1_promedio = sum(bleu1_por_ejemplo) / len(bleu1_por_ejemplo)

print(f"ROUGE-L promedio (sobre las 10.000): {rouge_l_promedio:.4f}")
print(f"BLEU-1 promedio (sobre las 10.000): {bleu1_promedio:.4f}")
print(f"(De referencia -- accuracy exacta ya calculada arriba: {accuracy:.4f}. Si estos tres "
      f"números son casi idénticos, es justo lo esperado -- ver la nota metodológica de arriba.)")

ROUGE-L promedio (sobre las 10.000): 0.5105
BLEU-1 promedio (sobre las 10.000): 0.3680
(De referencia -- accuracy exacta ya calculada arriba: 0.7389. Si estos tres números son casi idénticos, es justo lo esperado -- ver la nota metodológica de arriba.)


## 9c. Matriz de confusión

In [11]:
from sklearn.metrics import confusion_matrix

matriz_confusion = confusion_matrix(df_test['label'], df_test['prediccion'], labels=EMOCIONES)
df_matriz = pd.DataFrame(matriz_confusion, index=EMOCIONES, columns=EMOCIONES)
df_matriz.index.name = "real"
df_matriz.columns.name = "predicho"

RUTA_MATRIZ = f"07_matriz_confusion_{CLAVE_MODELO}.csv"
df_matriz.to_csv(RUTA_MATRIZ)
print(f"Matriz de confusión guardada en {RUTA_MATRIZ}")
df_matriz

Matriz de confusión guardada en 07_matriz_confusion_mistral_7b.csv


predicho,alegria,tristeza,enojo,miedo,sorpresa,burla,neutral
real,,,,,,,
alegria,2624,6,124,1,32,48,97
tristeza,14,342,50,3,3,1,10
enojo,115,36,2214,5,18,118,95
miedo,2,0,7,18,2,1,1
sorpresa,46,4,34,1,263,22,25
burla,76,8,139,1,14,1261,25
neutral,133,6,171,2,23,35,667


## 9d. Intervalos de confianza al 95% (bootstrapping) para accuracy y F1 macro

In [12]:
import numpy as np
from sklearn.metrics import f1_score

N_REMUESTREOS = 1000
rng = np.random.default_rng(42)

y_true_arr = df_test['label'].to_numpy()
y_pred_arr = df_test['prediccion'].to_numpy()
n = len(y_true_arr)

accs_boot = np.empty(N_REMUESTREOS)
f1s_boot = np.empty(N_REMUESTREOS)

for i in range(N_REMUESTREOS):
    idx = rng.integers(0, n, n)
    accs_boot[i] = accuracy_score(y_true_arr[idx], y_pred_arr[idx])
    f1s_boot[i] = f1_score(y_true_arr[idx], y_pred_arr[idx], labels=EMOCIONES, average='macro', zero_division=0)

accuracy_ic95 = (float(np.percentile(accs_boot, 2.5)), float(np.percentile(accs_boot, 97.5)))
f1_macro_ic95 = (float(np.percentile(f1s_boot, 2.5)), float(np.percentile(f1s_boot, 97.5)))

print(f"Accuracy: {accuracy:.4f}  IC 95%: [{accuracy_ic95[0]:.4f}, {accuracy_ic95[1]:.4f}]")
print(f"F1 macro: {reporte['macro avg']['f1-score']:.4f}  IC 95%: [{f1_macro_ic95[0]:.4f}, {f1_macro_ic95[1]:.4f}]")

Accuracy: 0.7389  IC 95%: [0.7303, 0.7473]
F1 macro: 0.7207  IC 95%: [0.6957, 0.7406]


## 10. Latencia de a un comentario a la vez (lote=1) y memoria GPU pico

In [13]:
import time

muestra_latencia = prompts[:N_MUESTRA_LATENCIA]

N_CALENTAMIENTO = 3
for p in muestra_latencia[:N_CALENTAMIENTO]:
    entrada = tokenizer(p, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(modelo.device)
    with torch.no_grad():
        modelo.generate(**entrada, max_new_tokens=6, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats()
inicio = time.time()
for p in muestra_latencia:
    entrada = tokenizer(p, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(modelo.device)
    with torch.no_grad():
        modelo.generate(**entrada, max_new_tokens=6, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
torch.cuda.synchronize()
fin = time.time()

latencia_prom_seg = (fin - inicio) / len(muestra_latencia)
memoria_pico_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

print(f"Latencia promedio por comentario: {latencia_prom_seg:.3f} s (sobre {len(muestra_latencia)} ejemplos, lote=1, "
      f"tras {N_CALENTAMIENTO} iteraciones de calentamiento)")
print(f"Memoria GPU pico durante inferencia: {memoria_pico_mb:.1f} MB")
print(f"Throughput sostenible aproximado: {1 / latencia_prom_seg:.2f} comentarios/segundo "
      f"(referencia indirecta para la afirmación de 'tiempo real' bajo ráfaga, ej. un gol)")

Latencia promedio por comentario: 1.672 s (sobre 300 ejemplos, lote=1, tras 3 iteraciones de calentamiento)
Memoria GPU pico durante inferencia: 4263.6 MB
Throughput sostenible aproximado: 0.60 comentarios/segundo (referencia indirecta para la afirmación de 'tiempo real' bajo ráfaga, ej. un gol)


## 11. Guardar la fila de este modelo, y las 10.000 predicciones individuales

In [14]:
import os

RUTA_RESULTADOS = "07_resultados_finales.csv"

fila = {
    "modelo": CLAVE_MODELO,
    "model_id": MODEL_ID,
    "checkpoint": ruta_relativa_adaptador,
    "n_test": len(df_test),
    "accuracy": accuracy,
    "accuracy_ic95_bajo": accuracy_ic95[0],
    "accuracy_ic95_alto": accuracy_ic95[1],
    "precision_macro": reporte["macro avg"]["precision"],
    "recall_macro": reporte["macro avg"]["recall"],
    "f1_macro": reporte["macro avg"]["f1-score"],
    "f1_macro_ic95_bajo": f1_macro_ic95[0],
    "f1_macro_ic95_alto": f1_macro_ic95[1],
    "precision_weighted": reporte["weighted avg"]["precision"],
    "recall_weighted": reporte["weighted avg"]["recall"],
    "f1_weighted": reporte["weighted avg"]["f1-score"],
    "pct_predicciones_invalidas": 100 * n_invalidas / len(df_test),
    "rouge_l_promedio": rouge_l_promedio,
    "bleu1_promedio": bleu1_promedio,
    "latencia_prom_seg": latencia_prom_seg,
    "latencia_lote_prom_seg": latencia_lote_prom_seg,
    "memoria_gpu_pico_mb": memoria_pico_mb,
    "almacenamiento_adaptador_mb": tamano_adaptador_mb,
}
for emocion in EMOCIONES:
    fila[f"f1_{emocion}"] = reporte.get(emocion, {}).get("f1-score", None)

df_fila = pd.DataFrame([fila])

if os.path.exists(RUTA_RESULTADOS):
    df_resultados = pd.read_csv(RUTA_RESULTADOS)
    df_resultados = df_resultados[df_resultados['modelo'] != CLAVE_MODELO]
    df_resultados = pd.concat([df_resultados, df_fila], ignore_index=True)
else:
    df_resultados = df_fila

df_resultados.to_csv(RUTA_RESULTADOS, index=False)
print(f"Guardado en {RUTA_RESULTADOS}. Modelos evaluados hasta ahora: {sorted(df_resultados['modelo'].tolist())}")

RUTA_PREDICCIONES = f"07_predicciones_{CLAVE_MODELO}.csv"
df_test[[COLUMNA_TEXTO, 'label', 'prediccion', 'prediccion_cruda', 'rouge_l', 'bleu1']].to_csv(
    RUTA_PREDICCIONES, index=False
)
print(f"Predicciones individuales ({len(df_test)}) guardadas en {RUTA_PREDICCIONES}")

df_resultados[['modelo', 'accuracy', 'f1_macro', 'latencia_prom_seg', 'memoria_gpu_pico_mb', 'almacenamiento_adaptador_mb']]

Guardado en 07_resultados_finales.csv. Modelos evaluados hasta ahora: ['mistral_7b']
Predicciones individuales (10000) guardadas en 07_predicciones_mistral_7b.csv


,modelo,accuracy,f1_macro,latencia_prom_seg,memoria_gpu_pico_mb,almacenamiento_adaptador_mb
0,mistral_7b,0.7389,0.72074,1.672464,4263.589355,164.25977


## 12. Subir `07_resultados_finales.csv` a Hugging Face

In [15]:
from huggingface_hub import upload_file, hf_hub_download as _hf_hub_download
from huggingface_hub.utils import EntryNotFoundError

try:
    ruta_remota_previa = _hf_hub_download(
        repo_id=REPO_DATOS, repo_type="dataset", filename="07_resultados_finales.csv"
    )
    df_remoto_previo = pd.read_csv(ruta_remota_previa)
    df_resultados = pd.concat([df_remoto_previo, df_resultados], ignore_index=True)
    df_resultados = df_resultados.drop_duplicates(subset="modelo", keep="last")
    df_resultados.to_csv(RUTA_RESULTADOS, index=False)
    print(f"Combinado con la versión previa en Hugging Face. Modelos en total: {sorted(df_resultados['modelo'].tolist())}")
except EntryNotFoundError:
    print("Todavía no existe una versión previa en Hugging Face -- se sube esta como la primera.")

upload_file(
    path_or_fileobj=RUTA_RESULTADOS,
    path_in_repo="07_resultados_finales.csv",
    repo_id=REPO_DATOS,
    repo_type="dataset",
)
print("07_resultados_finales.csv subido a tu bodega de datos en Hugging Face.")

upload_file(
    path_or_fileobj=RUTA_PREDICCIONES,
    path_in_repo=f"predicciones/{RUTA_PREDICCIONES}",
    repo_id=REPO_DATOS,
    repo_type="dataset",
)
upload_file(
    path_or_fileobj=RUTA_MATRIZ,
    path_in_repo=f"matrices_confusion/{RUTA_MATRIZ}",
    repo_id=REPO_DATOS,
    repo_type="dataset",
)
print("Predicciones individuales y matriz de confusión también subidas (carpetas 'predicciones/' y 'matrices_confusion/').")

07_resultados_finales.csv:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

Combinado con la versión previa en Hugging Face. Modelos en total: ['llama_3b', 'mistral_7b', 'phi35_mini', 'phi3_mini', 'qwen_1_5b', 'qwen_3b']
07_resultados_finales.csv subido a tu bodega de datos en Hugging Face.
Predicciones individuales y matriz de confusión también subidas (carpetas 'predicciones/' y 'matrices_confusion/').
